In [1162]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from glob import glob
import json

# Load TSV result files
tsv_files = glob("../results/**/*_ner_temporal_entity_scores.tsv", recursive=True)

def clean_model_name(raw_name):
    base = raw_name.replace("_ner_temporal_entity_scores.tsv", "")
    parts = base.split("_")
    try:
        model_type = parts[-1]
        model_size = parts[2] + '-' + parts[3]
        max_len = parts[-5]
        return f"{model_type}-{model_size}-{max_len}"
    except Exception:
        return raw_name

def get_model_type(raw_name):
    base = raw_name.replace("_ner_temporal_entity_scores.tsv", "")
    parts = base.split("_")
    try:
        index_run = parts.index("run")
        return parts[index_run + 1]
    except Exception:
        return raw_name

def get_max_len(raw_name):
    base = raw_name.replace("_ner_temporal_entity_scores.tsv", "")
    parts = base.split("_")
    try:
        return parts[-5]
    except Exception:
        return raw_name

def get_base_model(raw_name):
    base = raw_name.replace("_ner_temporal_entity_scores.tsv", "")
    parts = base.split("_")
    try:
        return '_'.join(parts[1:4])
    except Exception:
        return raw_name

# Load and annotate result TSVs
all_dfs = []
for file in tsv_files:
    df = pd.read_csv(file, sep="\t")
    df["run"] = os.path.basename(file).replace("_ner_temporal_entity_scores.tsv", "")
    df["run_clean"] = clean_model_name(df["run"].iloc[0])
    df["model"] = get_model_type(df["run"].iloc[0])
    df["max_len"] = get_max_len(df["run"].iloc[0])
    df["base_model"] = get_base_model(df["run"].iloc[0])
    all_dfs.append(df)

results_df = pd.concat(all_dfs, ignore_index=True)

# results_df[results_df["model"] == "time.baseline.True"] = "baseline"
# results_df[results_df["model"] == "time.baseline.False"] = "baseline"
# Load one matching JSON file from subfolder
csv_files = glob("../results/**/entity_statistics_test.tsv", recursive=True)
if csv_files:
    entity_stats_df = pd.read_csv(csv_files[0], sep="\t")
    entity_stats_df["entity_type"] = entity_stats_df["entity_type"].str.lower()

    # Compute micro average (sum of all entity counts per year for the test split)
micro_avg_df = (
    entity_stats_df[entity_stats_df["split"] == "test"]
    .groupby("year", as_index=False)["count"]
    .sum()
)
micro_avg_df["entity_type"] = "micro avg"
micro_avg_df["split"] = "test"

# Append to entity_stats_df
entity_stats_df = pd.concat([entity_stats_df, micro_avg_df], ignore_index=True)


In [1163]:
entity_stats_df

,split,year,entity_type,count
0,test,1798,loc,10
1,test,1798,org,4
2,test,1798,pers,30
3,test,1798,time,1
4,test,1808,loc,32
...,...,...,...,...
106,test,1978,micro avg,26
107,test,1988,micro avg,49
108,test,1998,micro avg,16
109,test,2008,micro avg,44


In [1164]:
entity_stats_df.year.unique()

array([1798, 1808, 1818, 1828, 1838, 1848, 1858, 1868, 1878, 1888, 1898,
       1908, 1918, 1928, 1938, 1948, 1958, 1968, 1978, 1988, 1998, 2008,
       2018])

In [1165]:
results_df.head()

,scope,time,entity_type,precision,recall,f1,run,run_clean,model,max_len,base_model
0,year,1798,loc,0.6364,0.8235,0.7179,model_dbmdz_bert_base_historic_multilingual_ca...,dev-bert-base-epochs,time-swiglu.concat.False,epochs,dbmdz_bert_base
1,year,1798,pers,0.4286,0.3750,0.4000,model_dbmdz_bert_base_historic_multilingual_ca...,dev-bert-base-epochs,time-swiglu.concat.False,epochs,dbmdz_bert_base
2,year,1798,micro avg,0.5862,0.6800,0.6296,model_dbmdz_bert_base_historic_multilingual_ca...,dev-bert-base-epochs,time-swiglu.concat.False,epochs,dbmdz_bert_base
3,year,1808,loc,0.9688,1.0000,0.9841,model_dbmdz_bert_base_historic_multilingual_ca...,dev-bert-base-epochs,time-swiglu.concat.False,epochs,dbmdz_bert_base
4,year,1808,pers,1.0000,0.8750,0.9333,model_dbmdz_bert_base_historic_multilingual_ca...,dev-bert-base-epochs,time-swiglu.concat.False,epochs,dbmdz_bert_base


In [1166]:
x = results_df[results_df['model'] == 'time.relative.True']
x = x[x['scope'] == "year"]
x[x['entity_type'] == 'micro avg']

,scope,time,entity_type,precision,recall,f1,run,run_clean,model,max_len,base_model


In [1167]:
x = results_df[results_df['model'] == 'time.relative.True']
x.scope.unique()

array([], dtype=object)

In [1168]:
results_df["time"].unique()

array(['1798', '1808', '1818', '1828', '1838', '1848', '1868', '1878',
       '1888', '1908', '1918', '1928', '1938', '1948', '1958', '1968',
       '1978', '1988', '1998', '2008', '1790s', '1800s', '1810s', '1820s',
       '1830s', '1840s', '1860s', '1870s', '1880s', '1900s', '1910s',
       '1920s', '1930s', '1940s', '1950s', '1960s', '1970s', '1980s',
       '1990s', '2000s', '1858', '1898', '2018', '1850s', '1890s',
       '2010s'], dtype=object)

In [1169]:
results_df.model.unique()

array(['time-swiglu.concat.False', 'time-swiglu.relative.True',
       'time-swiglu.baseline.True', 'time-swiglu.concat.True',
       'time-swiglu.late-cross-attention.False',
       'time-swiglu.adapter.False',
       'time-swiglu.late-cross-attention.True',
       'time-swiglu.adapter.True',
       'time-swiglu.early-cross-attention.True',
       'time-swiglu.baseline.False',
       'time-swiglu.early-cross-attention.False',
       'time-swiglu.relative.False'], dtype=object)

In [1170]:
results_df.head()

,scope,time,entity_type,precision,recall,f1,run,run_clean,model,max_len,base_model
0,year,1798,loc,0.6364,0.8235,0.7179,model_dbmdz_bert_base_historic_multilingual_ca...,dev-bert-base-epochs,time-swiglu.concat.False,epochs,dbmdz_bert_base
1,year,1798,pers,0.4286,0.3750,0.4000,model_dbmdz_bert_base_historic_multilingual_ca...,dev-bert-base-epochs,time-swiglu.concat.False,epochs,dbmdz_bert_base
2,year,1798,micro avg,0.5862,0.6800,0.6296,model_dbmdz_bert_base_historic_multilingual_ca...,dev-bert-base-epochs,time-swiglu.concat.False,epochs,dbmdz_bert_base
3,year,1808,loc,0.9688,1.0000,0.9841,model_dbmdz_bert_base_historic_multilingual_ca...,dev-bert-base-epochs,time-swiglu.concat.False,epochs,dbmdz_bert_base
4,year,1808,pers,1.0000,0.8750,0.9333,model_dbmdz_bert_base_historic_multilingual_ca...,dev-bert-base-epochs,time-swiglu.concat.False,epochs,dbmdz_bert_base


In [1171]:
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure matching types
# entity_stats_df["year"] = entity_stats_df["year"].astype(int)
# results_df["time"] = results_df["time"].astype(int)

for base_model in results_df["base_model"].unique():
    for max_len in results_df["max_len"].unique():
        for relative_year in results_df["relative_year"].unique():
            df = results_df[results_df["base_model"] == base_model]
            df = df[df["relative_year"] == relative_year]
            df = df[df["max_len"] == max_len]
            
        
            for entity in sorted(results_df["entity_type"].unique()):
                plt.figure(figsize=(14, 5))
        
                # Filter result data
                plot_data = df[(df["scope"] == "year") & (df["entity_type"] == entity)]
        
                if plot_data.empty:
                    continue  # Skip if there's nothing to plot
        
                sns.lineplot(
                    data=plot_data,
                    x="time", y="f1", hue="model", marker="o"
                )
        
                # Filter and annotate stats
                stats_filtered = entity_stats_df[
                    (entity_stats_df["entity_type"] == entity) & 
                    (entity_stats_df["split"] == "test") & 
                    (entity_stats_df["year"].isin(plot_data["time"]))
                ]
        
                for _, row in stats_filtered.iterrows():
                    plt.annotate(
                        f'n={row["count"]}', 
                        xy=(row["year"], 0.51), 
                        textcoords="offset points", 
                        xytext=(0, 5),
                        ha='center', fontsize=8, color='gray'
                    )
        
                plt.title(f"F1 Score by Year for Entity Type: {entity} (max_len={max_len})")
                plt.xticks(rotation=45)
                plt.ylim(0.5, 1.02)
                plt.grid(True)
                plt.tight_layout()
                plt.show()

KeyError: 'relative_year'

In [ ]:
entity_stats_df['year'].unique()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure matching types
# entity_stats_df["decade"] = entity_stats_df["year"].astype(int)
# entity_stats_df["decade"] = entity_stats_df["decade"].astype(int)
# results_df["time"] = results_df["time"].astype(int)


for max_len in results_df["max_len"].unique():
    df = results_df[results_df["scope"] == "decade"]
    df = df[df["max_len"] == max_len]
    

    for entity in sorted(results_df["entity_type"].unique()):
        plt.figure(figsize=(14, 5))

        # Filter result data
        plot_data = df[(df["scope"] == "decade") & (df["entity_type"] == entity)]

        if plot_data.empty:
            continue  # Skip if there's nothing to plot

        sns.lineplot(
            data=plot_data,
            x="time", y="f1", hue="model", marker="o"
        )

        # Filter and annotate stats
        # stats_filtered = entity_stats_df[
        #     (entity_stats_df["entity_type"] == entity) & 
        #     (entity_stats_df["split"] == "test") & 
        #     (entity_stats_df["decade"].isin(plot_data["time"]))
        # ]

        # for _, row in stats_filtered.iterrows():
        #     plt.annotate(
        #         f'n={row["count"]}', 
        #         xy=(row["decade"], 0.51), 
        #         textcoords="offset points", 
        #         xytext=(0, 5),
        #         ha='center', fontsize=8, color='gray'
        #     )

        plt.title(f"F1 Score by Year for Entity Type: {entity} (max_len={max_len})")
        plt.xticks(rotation=45)
        plt.ylim(0.5, 1.02)
        plt.grid(True)
        plt.tight_layout()
        plt.show()

In [ ]:
results_df["time"].unique()

In [ ]:
print(stats_filtered.head())
print(plot_data["time"].unique())

In [ ]:
results_df.head()

In [ ]:
results_df['relative_year'] = results_df['model'].str.extract(r'(True|False)$')[0]
results_df.tail()

In [ ]:
import matplotlib.pyplot as plt


for base_model in results_df["base_model"].unique():
    for max_len in results_df["max_len"].unique():
        for relative_year in results_df["relative_year"].unique():
            df = results_df[(results_df["max_len"] == max_len) & (results_df["base_model"] == base_model) & (results_df["relative_year"] == relative_year)]
    
            for entity in sorted(results_df["entity_type"].unique()):
                fig, ax1 = plt.subplots(figsize=(14, 5))
        
                # Plot F1 scores
                plot_data = df[(df["scope"] == "year") & (df["entity_type"] == entity)]
                if plot_data.empty:
                    continue
        
                sns.lineplot(data=plot_data, x="time", y="f1", hue="model", marker="o", ax=ax1)
                ax1.set_ylabel("F1 score")
                ax1.set_ylim(0.5, 1.02)
                # ax1.axhspan(0.5, 1.02, xmin=0, xmax=1, facecolor='red', alpha=0.05,
                #             label='Low-support region (n<15)')
                # Add grid + second y-axis for entity counts
                ax1.grid(True)
                ax2 = ax1.twinx()
        
                stats_filtered = entity_stats_df[
                    (entity_stats_df["entity_type"] == entity) & 
                    (entity_stats_df["split"] == "test") & 
                    (entity_stats_df["year"].isin(plot_data["time"]))
                ]
                stats_filtered = stats_filtered.sort_values("year")
        
                ax2.bar(stats_filtered["year"], stats_filtered["count"], alpha=0.2, color="gray")
                ax2.set_ylabel("Entity count")
                ax2.set_ylim(0, entity_stats_df["count"].max() * 1.1)
                for _, row in stats_filtered.iterrows():
                    ax2.text(row["year"], row["count"] + 3, f'n={row["count"]}', 
                             ha='center', fontsize=8, color='gray')
        
                plt.title(f"F1 Score and Entity Count by Year for Entity Type: {entity} (max_len={max_len}) (base_model={base_model}) (relative_year={relative_year})")
                plt.xticks(rotation=45)
                plt.tight_layout()
                plt.show()

In [ ]:
def classify_model_group(model_name):
    return "baseline" if "baseline" in model_name else "time-based"

results_df["model_group"] = results_df["model"].apply(classify_model_group)

grouped_df = (
    results_df[(results_df["scope"] == "year")]
    .groupby(["time", "entity_type", "model_group"], as_index=False)
    .agg({"f1": "mean"})
)


for entity in sorted(grouped_df["entity_type"].unique()):
    plot_data = grouped_df[grouped_df["entity_type"] == entity]
    if plot_data.empty:
        continue

    plt.figure(figsize=(14, 5))
    sns.lineplot(data=plot_data, x="time", y="f1", hue="model_group", marker="o")
    plt.title(f"Average F1 Score by Year — Entity: {entity} (Time-based vs Baseline)")
    plt.ylabel("F1 score")
    plt.ylim(0.5, 1.02)
    plt.grid(True)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
# Optional: Add boolean column for which group had higher mean F1 at each year/entity
def compare_groups(df):
    pivot = df.pivot(index="time", columns="model_group", values="f1")
    pivot["time-based_better"] = pivot["time-based"] > pivot["baseline"]
    return pivot

# Per entity-type comparison table
for entity in grouped_df["entity_type"].unique():
    print(f"\nEntity: {entity}")
    comparison = compare_groups(grouped_df[grouped_df["entity_type"] == entity])
    print(comparison[["time-based_better"]])

In [ ]:
def get_time_strategy(model_name):
    if "baseline" in model_name:
        return "baseline"
    elif model_name.startswith("time."):
        return model_name.replace(".True", "").replace(".False", "")
    else:
        return "other"

results_df["time_strategy"] = results_df["model"].apply(get_time_strategy)


time_methods_df = results_df[
    (results_df["scope"] == "year") &
    (results_df["time_strategy"] != "baseline")
]

avg_by_method = (
    time_methods_df
    .groupby(["entity_type", "time_strategy"], as_index=False)
    .agg(mean_f1=("f1", "mean"), std_f1=("f1", "std"))
    .sort_values("mean_f1", ascending=False)
)

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
sns.barplot(data=avg_by_method, x="mean_f1", y="time_strategy", hue="entity_type")
plt.xlabel("Average F1 Score")
plt.ylabel("Time Strategy")
plt.title("Average F1 Score per Time-Based Strategy by Entity Type")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Keep only time.relative and baseline models
filtered_df = results_df[
    results_df["model"].str.startswith("time.relative") |
    results_df["model"].str.contains("baseline")
]

# Make sure relative_year is extracted
if "relative_year" not in filtered_df.columns:
    filtered_df['relative_year'] = filtered_df['model'].str.extract(r'(True|False)$')[0]

# Plot per relative_year
for relative_year in ["True", "False"]:
    subset_df = filtered_df[filtered_df["relative_year"] == relative_year]

    for entity in sorted(subset_df["entity_type"].unique()):
        plot_data = subset_df[
            (subset_df["scope"] == "year") & 
            (subset_df["entity_type"] == entity)
        ]
        if plot_data.empty:
            continue

        plt.figure(figsize=(14, 5))
        sns.lineplot(data=plot_data, x="time", y="f1", hue="model", marker="o")
        plt.title(f"F1 Score by Year — Entity: {entity} | relative_year={relative_year}")
        plt.ylabel("F1 score")
        plt.ylim(0.5, 1.02)
        plt.grid(True)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

In [ ]:
filtered_df